# UA-FCCL: 100-split check and concept-lattice test (Colab)
Run the cells **one by one, from top to bottom**. If this Colab session was used before, first click **Runtime > Restart session**.

**Files needed on your computer:** `extra.py` (sent in chat) and `Shill Bidding Dataset.csv`.

- **Part A** repeats Shuttle, Yeast and Ecoli over 100 splits and prints rare-class coverage with 95% intervals (about 10 to 20 minutes).
- **Part B** builds a real concept lattice from the data (attribute values at or above a threshold theta count as present) and checks how well its concepts agree with the forest (about 15 to 30 minutes).

## Cell 1: get the repo, the clustered-conformal code and one package

In [ ]:
%cd /content
!rm -rf conformal-fuzzy-ccl class-conditional-conformal
!git clone -q https://github.com/mahibgm9676-lab/conformal-fuzzy-ccl.git
!git clone -q https://github.com/tiffanyding/class-conditional-conformal.git
%cd /content/conformal-fuzzy-ccl
!pip -q install ucimlrepo
import os
os.environ['CCL_DING_DIR'] = '/content/class-conditional-conformal'
!ls ccl_helpers.py

## Cell 2: upload `extra.py`
Click **Choose Files** and select the file.

In [ ]:
import glob, os, shutil
from google.colab import files
up = files.upload()
c = sorted(glob.glob('extra*.py'), key=os.path.getmtime)
if c[-1] != 'extra.py': shutil.copy(c[-1], 'extra.py')
print('using', c[-1])
!ls extra.py

## Cell 3: download the datasets (ignore the Shill Bidding message)

In [ ]:
!python 01_prepare_data.py
!ls data

## Cell 4: upload `Shill Bidding Dataset.csv`

In [ ]:
import shutil
from google.colab import files
up = files.upload()
for fn in up:
    shutil.move(fn, 'data/Shill Bidding Dataset.csv')
!ls data

## Cell 5: load the nine datasets

In [ ]:
import numpy as np, pandas as pd, time, os, warnings
warnings.filterwarnings('ignore')
import ccl_helpers as H
from extra import rare_check, summarise_rare, lattice_check
files_map = {'shill': 'Shill Bidding Dataset.csv', 'wilt': 'wilt.csv', 'htru2': 'htru2.csv', 'magic': 'magic.csv',
             'pageblocks': 'pageblocks.csv', 'drybean': 'drybean.csv', 'shuttle': 'shuttle.csv',
             'yeast': 'yeast.csv', 'ecoli': 'ecoli.csv'}
data = {n: H.load('data/' + f) for n, f in files_map.items()}
for n, (X, y) in data.items():
    print(n, X.shape, 'class sizes:', np.bincount(y).tolist())

## Part A, Cell 6: 100 splits for Shuttle, Yeast and Ecoli (results saved after each dataset)

In [ ]:
for n in ['ecoli', 'yeast', 'shuttle']:
    f = f'rare100_{n}.csv'
    if os.path.exists(f):
        print(n, 'already done, skipping'); continue
    t0 = time.time(); X, y = data[n]
    rare_check(n, X, y, range(100)).to_csv(f, index=False)
    print(n, 'done in', round((time.time() - t0) / 60, 1), 'min')

## Part A, Cell 7: summary with 95% intervals (bootstrap over splits)

In [ ]:
R = summarise_rare(pd.concat([pd.read_csv(f'rare100_{n}.csv') for n in ['ecoli', 'yeast', 'shuttle']]))
R.to_csv('rare100_summary.csv', index=False)
print(R.to_string(index=False))

## Part B, Cell 8: quick test on Ecoli (about 1 minute). Check that a table prints.
`n_concepts` is the number of formal concepts in the lattice (only counted for datasets with at most 10 attributes). `support` is the share of test cases whose attribute set matches at least one training object. `agree_forest_concept` is how often the majority class of the matching concept equals the forest decision; `agree_forest_pseudo` is the same for the nearest pseudo-concept on the same cases.

In [ ]:
t0 = time.time(); X, y = data['ecoli']
d = lattice_check('ecoli', X, y, range(2))
print('time', round(time.time() - t0, 1), 's')
print(d.round(3).to_string(index=False))

## Part B, Cell 9: all nine datasets, 5 splits each (results saved after each dataset)

In [ ]:
for n in files_map:
    f = f'lattice_{n}.csv'
    if os.path.exists(f):
        print(n, 'already done, skipping'); continue
    t0 = time.time(); X, y = data[n]
    lattice_check(n, X, y, range(5)).to_csv(f, index=False)
    print(n, 'done in', round((time.time() - t0) / 60, 1), 'min')

## Part B, Cell 10: summary

In [ ]:
L = pd.concat([pd.read_csv(f'lattice_{n}.csv') for n in files_map])
S = L.groupby(['dataset', 'theta']).agg(n_concepts=('n_concepts', 'max'), support=('support', 'mean'),
        mean_extent=('mean_extent', 'mean'), purity=('purity', 'mean'),
        agree_concept=('agree_forest_concept', 'mean'), agree_pseudo=('agree_forest_pseudo', 'mean')).round(3)
S.to_csv('lattice_summary.csv')
pd.set_option('display.width', 220); pd.set_option('display.max_rows', 500)
print(S.to_string())

## Cell 11: download everything and send it to me

In [ ]:
!zip -q extra_results.zip rare100_*.csv lattice_*.csv
from google.colab import files
files.download('extra_results.zip')